In [7]:
import os
import schema
import pandas as pd
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt


sc.set_figure_params(scanpy = False)
plt.rcParams["font.family"] = "Arial"
workdir = "./ComicGTN/data/BMMC-bench-1"
RNA_seq = sc.read(os.path.join(workdir, "Gene_Cell.mtx"))
ATAC_seq = sc.read(os.path.join(workdir, "Peak_Cell.mtx"))
Cell_names = pd.read_csv(os.path.join(workdir, "Cell_names.tsv"), sep = "\t", header = None)
Cell_types = pd.read_csv(os.path.join(workdir, "Cell_types.tsv"), sep = "\t", header = None)
Gene_names = pd.read_csv(os.path.join(workdir, "Gene_names.tsv"), sep = "\t", header = None)
RNA_count = RNA_seq.X
adata =  ad.AnnData(RNA_count.transpose(), dtype = "int32")
adata.obs_names = Cell_names[0]
adata.var_names = Gene_names[0]
adata.var_names_make_unique()
adata.uns["atac.X"] = ATAC_seq.X.transpose()

/home/jsl/anaconda3/envs/Schema/lib/python3.10/site-packages/anndata/_core/anndata.py:381: FutureWarning: The dtype argument is deprecated and will be removed in late 2024.
  warnings.warn(
/home/jsl/anaconda3/envs/Schema/lib/python3.10/site-packages/anndata/_core/anndata.py:812: UserWarning: 
AnnData expects .obs.index to contain strings, but got values like:
    [0, 1, 2, 3, 4]

    Inferred to be: integer

  names = self._prep_dim_index(names, "obs")


In [8]:
from sklearn.decomposition import TruncatedSVD
svd2 = TruncatedSVD(n_components = 50, random_state = 17)
H2 = svd2.fit_transform(adata.uns["atac.X"])

In [9]:
sqp99 = schema.SchemaQP(0.99, mode = 'affine', params = {"decomposition_model":"nmf",
                                                      "num_top_components":50,
                                                      "do_whiten": 0,
                                                      "dist_npairs": 5000000})
dz99 = sqp99.fit_transform(adata.X, [H2], ['feature_vector'], [1])

Running change-of-basis transform (nmf, 50 components)... done.
Running quadratic program... done.


In [10]:
import schema.utils
fcluster = schema.utils.get_leiden_clustering #feel free to try your own clustering algo

# ld_cluster_rna = fcluster(sqp99._decomp_mdl.transform(adata.X.todense()))
ld_cluster_atac = fcluster(H2)
ld_cluster_sqp99 = fcluster(dz99)
pd.DataFrame(ld_cluster_sqp99).to_csv(os.path.join(workdir, "Schema_pred.tsv"), sep = "\t", header = False, index = False)